In [21]:
#%pip install scikit-learn

import sys
import os
import re
import warnings
import urllib3
import urllib.parse
from configparser import ConfigParser
from datetime import date, datetime, timedelta, UTC
from concurrent.futures import ThreadPoolExecutor, as_completed

import numpy as np
import pandas as pd

warnings.simplefilter(action='ignore', category=pd.errors.SettingWithCopyWarning)

# Add your local ThreatConnect SDK to path
sys.path.append(r"Z:\HTOC\Data_Analytics\threatconnect")
from ThreatConnect import ThreatConnect
from RequestObject import RequestObject
from Owners import Owners

# Add your project repo to path
project_root = r"H:\HTOC\scripts\Data Movement\ThrearConnect-api-pull"
if project_root not in sys.path:
    sys.path.append(project_root)

from utils.config_loader import load_config

# Load API config
config_path = os.path.join(project_root, "utils", "config.json")
try:
    api_secret_key, api_access_id, api_base_url, api_default_org = load_config(config_path)
    display(f"Loaded config from: {config_path}")
    display(f"Base URL: {api_base_url}")
    display(f"Access ID: {api_access_id}")
    display(f"Default Org: {api_default_org}")
except Exception as e:
    display(f"[ERROR] Failed to load configuration: {e}")
    sys.exit(1)

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

try:
    tc = ThreatConnect(api_access_id, api_secret_key, api_default_org, api_base_url)
    display("ThreatConnect initialized.")
except Exception as e:
    display(f"[ERROR] Failed to initialize ThreatConnect: {e}")
    sys.exit(1)




'Loaded config from: H:\\HTOC\\scripts\\Data Movement\\ThrearConnect-api-pull\\utils\\config.json'

'Base URL: https://hvs.threatconnect.com/api'

'Access ID: 09783848890162390382'

'Default Org: HTOC Org'

'ThreatConnect initialized.'

In [36]:
import pandas as pd
from datetime import datetime, timedelta
import pytz
import urllib.parse

# Configuration for ThreatConnect indicator query
QUERY_LOOKBACK_DAYS = 21  # days of lastObserved activity to include
INDICATOR_TYPE_NAMES = [
    "URL"
]
OWNER_NAMES = [
    'HTOC Org'
]
RESULT_PAGE_SIZE = 500  # keep this smaller; same fields, just paged

# Setup
cutoff = pd.Timestamp.utcnow()
start_date = (datetime.now(pytz.UTC) - timedelta(days=QUERY_LOOKBACK_DAYS)).date()
start = f"{start_date}T00:00:00Z"

type_names = INDICATOR_TYPE_NAMES
type_name_condition = ", ".join([f'"{t}"' for t in type_names])

list_of_owners = OWNER_NAMES

# Build owner IN (...) clause
owner_condition = ", ".join([f'"{o}"' for o in list_of_owners])

tql_raw = (
    f'ownerName IN ({owner_condition}) AND '
    f'typeName IN ({type_name_condition}) AND '
    f'dateAdded >= "2026-06-24" AND '
    f'dateAdded <= "2026-07-08" AND '
    'summary CONTAINS "tcp://"'
)

tql_encoded = urllib.parse.quote(tql_raw)

final_results = []

# Query indicators (paginate so you don't 502 with heavy fields)
# Create a NEW RequestObject WITHOUT owner restriction to query across all owners
ro_multi = RequestObject()
ro_multi.set_http_method('GET')

result_start = 0
result_limit = RESULT_PAGE_SIZE

while True:
    try:
        # NOTE: same fields list you requested (tags,observations,associatedGroups,falsePositives,threatAssess)
        # Only change here is removing the trailing comma after threatAssess which can break parsing.
        ro_multi.set_request_uri(
            f'/v3/indicators?tql={tql_encoded}'
            f'&fields=tags,observations,associatedGroups,falsePositives,threatAssess'
            f'&resultStart={result_start}&resultLimit={result_limit}'
        )

        response = tc.api_request(ro_multi)

        ct = response.headers.get('content-type', '')
        if not ct.startswith('application/json'):
            raise RuntimeError(f"Non-JSON response ({ct}): {response.content[:200]}")

        results = response.json()
        data_items = results.get('data', []) or []

        # stop when no more results
        if not data_items:
            break

        final_results.append(results)
        result_start += result_limit

    except Exception as e:
        display(f"Failed to query indicators (start={result_start}): {e}")
        break

# Normalize results
normalized_data = []
for result in final_results:
    data_items = result.get('data', [])
    if not data_items:
        display("No data returned in API response:", result)
    for item in data_items:
        if isinstance(item, dict) and 'summary' in item:
            if item.get('id') is not None:
                item = {**item, 'id': str(item['id'])}
            normalized_data.append(item)

if normalized_data:
    observed_src = pd.json_normalize(normalized_data)
    observed_src['indicator'] = observed_src['summary'].astype(str).str.split().str[0].str.strip()
    
    # Create a 'sources' column by aggregating ownerName values per indicator
    sources_per_indicator = (
        observed_src.groupby('indicator')['ownerName']
        .apply(lambda x: ', '.join(sorted(set(x))))
        .reset_index()
        .rename(columns={'ownerName': 'sources'})
    )

    # Merge sources back into observed_src
    observed_src = observed_src.merge(sources_per_indicator, on='indicator', how='left')
    # Filter to keep only records where ownerName is 'HTOC Org'
    observed_src = observed_src[observed_src['ownerName'] == 'HTOC Org'].copy()
else:
    display("No valid indicator data found.")
    observed_src = pd.DataFrame()

# Exclude rows where the 'tags.data' key is present (not None/NaN) in observed_src
if "tags.data" in observed_src.columns:
    observed_src = observed_src[observed_src["tags.data"].isna() | observed_src["tags.data"].isnull()]

if "description" in observed_src.columns:
    observed_src = observed_src[observed_src["description"].isna() | observed_src["description"].isnull()]

display(observed_src)

,id,dateAdded,ownerId,ownerName,webLink,type,lastModified,rating,confidence,threatAssessRating,...,privateFlag,active,activeLocked,text,legacyLink,description,tags.data,associatedGroups.data,indicator,sources
0,21392098237000749,2026-06-26T00:00:36Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,URL,2026-07-08T13:06:53Z,5.0,100,5.0,...,False,True,False,tcp://cl0p.annievy.com:2096,https://hvs.threatconnect.com/auth/indicators/...,NaN,NaN,NaN,tcp://cl0p.annievy.com:2096,HTOC Org
1,22517998141002164,2026-06-26T00:14:19Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,URL,2026-07-07T20:57:53Z,5.0,100,4.5,...,False,True,False,tcp://stuxnet.jav69.blog:22,https://hvs.threatconnect.com/auth/indicators/...,NaN,NaN,NaN,tcp://stuxnet.jav69.blog:22,HTOC Org
2,22517998141002163,2026-06-26T00:14:19Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,URL,2026-07-07T20:57:53Z,5.0,100,4.5,...,False,True,False,tcp://stuxnet.jav69.blog:443,https://hvs.threatconnect.com/auth/indicators/...,NaN,NaN,NaN,tcp://stuxnet.jav69.blog:443,HTOC Org
3,22517998141002162,2026-06-26T00:14:19Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,URL,2026-07-07T20:57:53Z,5.0,100,4.5,...,False,True,False,tcp://stuxnet.jav69.blog:2086,https://hvs.threatconnect.com/auth/indicators/...,NaN,NaN,NaN,tcp://stuxnet.jav69.blog:2086,HTOC Org
4,22517998141002160,2026-06-26T00:14:18Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,URL,2026-07-07T20:57:53Z,5.0,100,4.5,...,False,True,False,tcp://stuxnet.jav69.blog:2087,https://hvs.threatconnect.com/auth/indicators/...,NaN,NaN,NaN,tcp://stuxnet.jav69.blog:2087,HTOC Org
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
23849,21392098236019719,2026-06-24T12:30:36Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,URL,2026-06-24T20:40:13Z,5.0,100,4.5,...,False,True,False,tcp://iloveyou.sexphimhayp.pro:2083,https://hvs.threatconnect.com/auth/indicators/...,NaN,NaN,NaN,tcp://iloveyou.sexphimhayp.pro:2083,HTOC Org
23850,21392098236019718,2026-06-24T12:30:36Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,URL,2026-06-24T20:40:13Z,5.0,100,4.5,...,False,True,False,tcp://nanocore.sexphimhayp.pro:8443,https://hvs.threatconnect.com/auth/indicators/...,NaN,NaN,NaN,tcp://nanocore.sexphimhayp.pro:8443,HTOC Org
23851,21392098236019717,2026-06-24T12:30:36Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,URL,2026-06-24T20:40:13Z,5.0,100,4.5,...,False,True,False,tcp://nanocore.sexphimhayp.pro:2095,https://hvs.threatconnect.com/auth/indicators/...,NaN,NaN,NaN,tcp://nanocore.sexphimhayp.pro:2095,HTOC Org
23852,21392098236019716,2026-06-24T12:30:36Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,URL,2026-06-24T20:40:13Z,5.0,100,4.5,...,False,True,False,tcp://nanocore.sexphimhayp.pro:22,https://hvs.threatconnect.com/auth/indicators/...,NaN,NaN,NaN,tcp://nanocore.sexphimhayp.pro:22,HTOC Org


In [23]:
import pandas as pd
import urllib.parse

# Lookup by URL-encoded summary (TC ids can lose precision as float64 in pandas)
indicator_summaries = (
    observed_src['summary']
    .dropna()
    .astype(str)
    .str.strip()
    .unique()
)

attributes_data = []
indicators_with_no_attributes = []
ro = RequestObject()

for summary in indicator_summaries:
    encoded = urllib.parse.quote(summary, safe='')

    try:
        ro.set_http_method('GET')
        ro.set_request_uri(
            f'/v3/indicators/{encoded}?fields=attributes&resultStart=0&resultLimit=1000'
        )
        response = tc.api_request(ro, log=False)

        ct = response.headers.get('content-type', '')
        if not ct.startswith('application/json'):
            indicators_with_no_attributes.append(summary)
            continue

        body = response.json() or {}
        if body.get('status') == 'Error':
            indicators_with_no_attributes.append(summary)
            continue

        data = body.get('data', {}) or {}
        attributes = (data.get('attributes') or {}).get('data', []) or []

        if not attributes:
            indicators_with_no_attributes.append(summary)
        else:
            for attr in attributes:
                attr['indicator_id'] = data.get('id')
                attr['summary'] = data.get('summary') or summary
                attr['indicator_type'] = data.get('type')
                attr['ownerName'] = data.get('ownerName')
                attributes_data.append(attr)

    except RuntimeError:
        indicators_with_no_attributes.append(summary)
    except Exception:
        indicators_with_no_attributes.append(summary)

attributes_observed_src = pd.json_normalize(attributes_data) if attributes_data else pd.DataFrame()

if not attributes_observed_src.empty and 'id' in attributes_observed_src.columns:
    attributes_observed_src = attributes_observed_src.drop_duplicates(subset='id').reset_index(drop=True)

print(f'Indicators queried: {len(indicator_summaries):,}')
print(f'Indicators with attributes: {len(indicator_summaries) - len(indicators_with_no_attributes):,}')
print(f'Attribute rows: {len(attributes_observed_src):,}')

if not attributes_observed_src.empty and 'summary' in attributes_observed_src.columns:
    filtered_with_attrs = observed_src[observed_src['summary'].isin(attributes_observed_src['summary'])]
else:
    filtered_with_attrs = observed_src.iloc[0:0]

no_attrs_df = observed_src[observed_src['summary'].isin(indicators_with_no_attributes)]

filtered_recent_tags = pd.concat([filtered_with_attrs, no_attrs_df], ignore_index=True)
filtered_recent_tags = filtered_recent_tags.drop_duplicates(subset='summary').reset_index(drop=True)

display(attributes_observed_src.head(20))
display(filtered_recent_tags.head(20))


KeyboardInterrupt: 